In [7]:
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split, StratifiedKFold,GridSearchCV
from sklearn.metrics import mean_squared_error, make_scorer, r2_score, confusion_matrix
import numpy as np
from sklearn.decomposition import PCA
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import RFECV
from sklearn.model_selection import KFold
from sklearn.metrics import precision_score, recall_score

In [8]:
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

In [9]:
df = pd.read_excel('/content/all_data_0710.xlsx')

# Clean Data
df=df.drop(['accident level','fatality','损伤位置','损伤种类'],axis=1)
df=df.dropna(axis=0)


In [10]:
from transformers import BertTokenizer, BertModel
import torch
tokenizer = BertTokenizer.from_pretrained("bert-base-chinese")
model = BertModel.from_pretrained("bert-base-chinese")

def get_embedding(text):
    # Tokenize the text
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outputs = model(**inputs)
    # Use the mean pooling of the last hidden state for sentence-level embeddings
    embedding = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
    return embedding


In [12]:
Cause_Embeddings = df['Cause of Accident'].fillna("").apply(lambda x: get_embedding(x))
Traffic_Embeddings=df['事发水域路况'].fillna("").apply(lambda x: get_embedding(x))

df['Bert_cause'],df['Bert_traffic']=Cause_Embeddings,Traffic_Embeddings

In [ ]:
# Create target variable
y = df["accident type"].astype(str)

# Create target to label mapping
y_encoded, y_labels = pd.factorize(y)
target_mapping = dict(zip(range(len(y_labels)), y_labels))

# Drop the original target column
df=df.drop(["accident type"],axis=1)

# One-hot encode the remaining features
df_encoded = pd.get_dummies((df), drop_first=True)

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(df_encoded, y_encoded, test_size=0.3, random_state=42)


labels_in_data = sorted(np.unique(y_test))

# parameter thoice and models
models = {
    'Gradient Boosting': (GradientBoostingClassifier(), {
        'clf__n_estimators': [100, 200],
        'clf__max_depth': [3, 5],
    }),
    'Random Forest': (RandomForestClassifier(), {
        'clf__n_estimators': [100, 200],
        'clf__max_depth': [None, 10],
    }),
    'SVM': (SVC(), {
        'clf__C': [1, 10],
        'clf__kernel': ['linear', 'rbf'],
    }),
    'KNN': (KNeighborsClassifier(), {
        'clf__n_neighbors': [3, 5, 7],
    }),
}

# save results
results = {}

# Train each model with GridSearchCV
for name, (model, param_grid) in models.items():
    pipe = Pipeline([
        ('clf', model)
    ])
    grid = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train, y_train)

    y_pred = grid.best_estimator_.predict(X_test)
    results[name] = {
        'Best Parameters': grid.best_params_,
        'Test Accuracy': grid.best_estimator_.score(X_test, y_test),
        'Classification Report': classification_report(
    y_test, y_pred,
    labels=labels_in_data,
    target_names = [target_mapping[i] for i in labels_in_data if i in target_mapping]
)
    }

# Print results
for model_name, metrics in results.items():
    print(f"\n=== {model_name} ===")
    print("Best Parameters:", metrics['Best Parameters'])
    print("Test Accuracy:", metrics['Test Accuracy'])
    print("Classification Report:\n", metrics['Classification Report'])

In [ ]:
model = GradientBoostingClassifier()
model.fit(X_train,y_train)  # y_encoded is from your pd.factorize

# Get feature importances
importances = model.feature_importances_
feature_names = X_train.columns

# Create a DataFrame for easy viewing
feat_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False)

print(feat_importance_df)